# 🔍 Khám phá chuyên sâu: Trích xuất các dòng dữ liệu bị thiếu

Trước khi quyết định xóa (drop) hay điền (fill), chúng ta sẽ lọc riêng các trường hợp bị thiếu dữ liệu ra các file CSV nhỏ hoặc hiển thị trực tiếp để kiểm tra thủ công (cross-check).


In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

# Đường dẫn tới file dữ liệu
DATA_PATH = "../data/interim/latest_standardized_data.csv"
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print(f"✅ Đã tải dữ liệu: {df_raw.shape[0]:,} hàng × {df_raw.shape[1]} cột")

# Tạo thư mục để lưu các file CSV trích xuất
extract_dir = "../data/interim/missing_cases_extracted"
os.makedirs(extract_dir, exist_ok=True)
print(f"Đã tạo thư mục lưu trữ các case thiếu: {extract_dir}")

✅ Đã tải dữ liệu: 35,278 hàng × 9 cột
Đã tạo thư mục lưu trữ các case thiếu: ../data/interim/missing_cases_extracted


---
## 1. Thống kê tổng quan tỷ lệ thiếu dữ liệu của toàn bộ dataset


In [2]:
missing_stats = pd.DataFrame({
    'Số dòng thiếu': df_raw.isnull().sum(),
    'Tỷ lệ (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
missing_stats = missing_stats[missing_stats['Số dòng thiếu'] > 0].sort_values(by='Tỷ lệ (%)', ascending=False)

print("TỔNG HỢP CÁC TRƯỜNG DỮ LIỆU BỊ THIẾU:")
display(missing_stats)

TỔNG HỢP CÁC TRƯỜNG DỮ LIỆU BỊ THIẾU:


,Số dòng thiếu,Tỷ lệ (%)
change_commit_date,22282,63.16
change_commit_url,22282,63.16
commit_date,1203,3.41
commit_url,1203,3.41


---
## 2. Phân tích CASE 1: Thiếu `commit_date` (~8%)
Giả thuyết: Lỗi 404 (Repo bị xóa, đổi chủ, hoặc để chế độ private). Chúng ta sẽ trích xuất danh sách này ra để bạn có thể truy cập thử vào URL xem có đúng là 404 không.

In [3]:
# Lọc các dòng thiếu commit_date
df_missing_commit_date = df_raw[df_raw['commit_date'].isnull()]

print(f"Số dòng thiếu commit_date: {len(df_missing_commit_date):,} dòng")

# Xem thử 5 dòng đầu (chỉ hiển thị các cột quan trọng để check URL)
cols_to_show = ['id', 'repo', 'commit_url']
existing_cols = [c for c in cols_to_show if c in df_missing_commit_date.columns]

print("\nMột vài sample để check thủ công:")
display(df_missing_commit_date[existing_cols].head(5))

# Xuất ra file CSV để team cùng kiểm tra
out_path_1 = os.path.join(extract_dir, "missing_commit_date.csv")
df_missing_commit_date.to_csv(out_path_1, index=False)
print(f"\n💾 Đã trích xuất toàn bộ {len(df_missing_commit_date)} dòng ra file: {out_path_1}")

Số dòng thiếu commit_date: 1,203 dòng

Một vài sample để check thủ công:


,id,repo,commit_url
81,js__ai_created__588714688__7961ed1d86c9__204::...,harshitethic/Openai-clone,NaN
154,js__generated_by_chatgpt__696458575__078fdd3ab...,arctyx-0/Bae-is-very-stinky,NaN
293,python__generated_by_copilot__698621599__8b0f6...,Kreyren/shellcheck-bug-arrays-in-posix,NaN
306,python__suggested_by_copilot__698621599__8b0f6...,Kreyren/shellcheck-bug-arrays-in-posix,NaN
331,python__created_using_chatgpt__606835477__4378...,xieliaing/Side-Projects,NaN



💾 Đã trích xuất toàn bộ 1203 dòng ra file: ../data/interim/missing_cases_extracted/missing_commit_date.csv


---
## 3. Phân tích CASE 2: Thiếu `change_commit_date` (~70%)
Giả thuyết: Code AI tạo ra KHÔNG bị sửa lại. Trích xuất ra để đọc thử các `code_block` xem có đúng là các đoạn code đã hoàn thiện và không cần refactor không.

In [4]:
if 'change_commit_date' in df_raw.columns:
    # Lọc các dòng thiếu change_commit_date
    df_missing_change = df_raw[df_raw['change_commit_date'].isnull()]
    
    print(f"Số dòng thiếu change_commit_date: {len(df_missing_change):,} dòng")
    
    # Xem thử 5 dòng đầu
    cols_to_show_2 = ['id', 'language', 'file', 'code_block', 'commit_url']
    existing_cols_2 = [c for c in cols_to_show_2 if c in df_missing_change.columns]
    
    print("\nMột vài sample (Code AI chưa bị sửa):")
    display(df_missing_change[existing_cols_2].head(5))
    
    # Xuất ra file CSV
    out_path_2 = os.path.join(extract_dir, "missing_change_commit_date.csv")
    df_missing_change.to_csv(out_path_2, index=False)
    print(f"\n💾 Đã trích xuất toàn bộ {len(df_missing_change)} dòng ra file: {out_path_2}")
else:
    print("Không tìm thấy cột 'change_commit_date'.")

Số dòng thiếu change_commit_date: 22,282 dòng

Một vài sample (Code AI chưa bị sửa):


,id,language,file,code_block,commit_url
0,python__written_by_gpt__433861003__e397ad21a6e...,python,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,https://github.com/flowerett/aoc/commit/52a0fb...
1,python__written_by_gpt__433861003__e397ad21a6e...,python,2022/python/d1-gpt.py,# solution written by GPT chatbot\n# works and...,https://github.com/flowerett/aoc/commit/52a0fb...
2,python__written_by_ai__437874902__14f89ee997ec...,python,2017/puzzle2.py,# Written by AI\n\ndef spreadsheet_checksum(li...,https://github.com/tipa16384/adventofcode/comm...
3,python__written_by_ai__437874902__14f89ee997ec...,python,2017/puzzle2.py,# Written by AI\n\ndef spreadsheet_checksum(li...,https://github.com/tipa16384/adventofcode/comm...
4,python__written_by_ai__437874902__1d766ed4953e...,python,2017/puzzle1.py,"# Written by AI\n\nwith open('puzzle1.txt', 'r...",https://github.com/tipa16384/adventofcode/comm...



💾 Đã trích xuất toàn bộ 22282 dòng ra file: ../data/interim/missing_cases_extracted/missing_change_commit_date.csv


---
## 4. Các trường hợp missing hỗn hợp (Missing nhiều cột)
Kiểm tra xem có dòng nào vừa thiếu `commit_date`, vừa thiếu thông tin repo hay ngôn ngữ hay không.

In [5]:
# Tính số lượng giá trị Na trong mỗi dòng
df_raw['missing_count'] = df_raw.isnull().sum(axis=1)

# Lọc ra những dòng missing từ 3 cột trở lên (nếu có)
df_heavy_missing = df_raw[df_raw['missing_count'] >= 3]

print(f"Có {len(df_heavy_missing):,} dòng bị thiếu từ 3 trường dữ liệu trở lên.")

if len(df_heavy_missing) > 0:
    display(df_heavy_missing.head(5))
    out_path_3 = os.path.join(extract_dir, "heavy_missing_rows.csv")
    df_heavy_missing.drop(columns=['missing_count']).to_csv(out_path_3, index=False)
    print(f"\n💾 Đã trích xuất ra file: {out_path_3}")

# Dọn dẹp lại dataframe
df_raw = df_raw.drop(columns=['missing_count'])

Có 878 dòng bị thiếu từ 3 trường dữ liệu trở lên.


,id,repo,language,commit_url,file,code_block,commit_date,change_commit_url,change_commit_date,missing_count
154,js__generated_by_chatgpt__696458575__078fdd3ab...,arctyx-0/Bae-is-very-stinky,js,NaN,js/dynamicmessage.js,// This JavaScript code defines a function cal...,NaN,NaN,NaN,4
331,python__created_using_chatgpt__606835477__4378...,xieliaing/Side-Projects,python,NaN,Side-Project/Youtube/YouTube2MP3.py,"# created using ChatGPT on Feb 18,2022\n# with...",NaN,NaN,NaN,4
332,python__created_using_chatgpt__606835477__4378...,xieliaing/Side-Projects,python,NaN,Side-Project/Youtube/YouTube2MP3.py,"# created using ChatGPT on Feb 18,2022\n# with...",NaN,NaN,NaN,4
880,python__ai_created__519433264__5b9b8a934052__4...,mahsanghani/NLP,python,NaN,LLM/chat2.py,import os\nimport openai\n\nopenai.api_key = o...,NaN,NaN,NaN,4
1049,python__ai_created__659423764__da3f09ff5230__9...,javovelez/langchain_test,python,NaN,zero_to_hero/03_using_memory.py,import os\nfrom langchain.llms import OpenAI\n...,NaN,NaN,NaN,4



💾 Đã trích xuất ra file: ../data/interim/missing_cases_extracted/heavy_missing_rows.csv
